In [7]:
from data_frame.sql_executor import SQLExecutor
from data_frame.spark_utils import get_spark
from data_frame.view.view_manager import ViewManager

In [8]:
spark = get_spark(app_name="SQL Queries Execution")

In [9]:

# Create sample data
sales_data = [
    (1, "2024-01-01", 100, "North", "Electronics"),
    (2, "2024-01-02", 150, "South", "Clothing"),
    (3, "2024-01-03", 200, "North", "Electronics"),
    (4, "2024-01-04", 120, "East", "Clothing"),
    (5, "2024-01-05", 180, "North", "Furniture")
]
df_sales = spark.createDataFrame(sales_data, 
                                 ["sale_id", "date", "amount", "region", "category"])

## 1. Parameterized Queries

In [10]:
ViewManager.create_temp_view(df_sales, "sales")

# Parameterized query
params = {
    "region": "North",
    "min_amount": 150
}

query = """
    SELECT category, COUNT(*) as sales_count, SUM(amount) as total_amount
    FROM sales
    WHERE region = '${region}' AND amount >= ${min_amount}
    GROUP BY category
    ORDER BY total_amount DESC
"""

result = SQLExecutor.execute_query(spark, query, params)
print("Parameterized query result:")
result.show()

Executing SQL Query:

    SELECT category, COUNT(*) as sales_count, SUM(amount) as total_amount
    FROM sales
    WHERE region = 'North' AND amount >= 150
    GROUP BY category
    ORDER BY total_amount DESC

Parameterized query result:
+-----------+-----------+------------+
|   category|sales_count|total_amount|
+-----------+-----------+------------+
|Electronics|          1|         200|
|  Furniture|          1|         180|
+-----------+-----------+------------+



## 2. Multiple Queries

In [11]:
queries = [
    "SELECT COUNT(*) as total_sales FROM sales",
    "SELECT AVG(amount) as avg_amount FROM sales",
    "SELECT region, SUM(amount) as total FROM sales GROUP BY region"
]

results = SQLExecutor.execute_multi_query(spark, queries)
print("Multiple query results:")
for i, df in enumerate(results):
    print(f"\nQuery {i+1}:")
    df.show()

Multiple query results:

Query 1:
+-----------+
|total_sales|
+-----------+
|          5|
+-----------+


Query 2:
+----------+
|avg_amount|
+----------+
|     150.0|
+----------+


Query 3:
+------+-----+
|region|total|
+------+-----+
| North|  480|
| South|  150|
|  East|  120|
+------+-----+



## 3. Common Table Expressions (CTEs)

In [12]:
ctes = {
    "region_totals": """
        SELECT region, SUM(amount) as region_total
        FROM sales
        GROUP BY region
    """,
    "overall_stats": """
        SELECT AVG(amount) as overall_avg, MAX(amount) as overall_max
        FROM sales
    """
}

main_query = """
    SELECT r.*, 
           o.overall_avg,
           o.overall_max,
           ROUND(r.region_total / o.overall_avg, 2) as ratio_to_avg
    FROM region_totals r
    CROSS JOIN overall_stats o
    ORDER BY region_total DESC
"""

cte_result = SQLExecutor.execute_with_cte(spark, ctes, main_query)
print("CTE query result:")
cte_result.show()

CTE query result:
+------+------------+-----------+-----------+------------+
|region|region_total|overall_avg|overall_max|ratio_to_avg|
+------+------------+-----------+-----------+------------+
| North|         480|      150.0|        200|         3.2|
| South|         150|      150.0|        200|         1.0|
|  East|         120|      150.0|        200|         0.8|
+------+------------+-----------+-----------+------------+

